词嵌入 (Embedding)
给每个词分配一个固定长度的连续向量
猫 = [0.12, -0.59, 0.88, ...]
狗 = [0.15, -0.55, 0.82, ...]（数值非常接近）

In [3]:
import torch
import torch.nn as nn

# 1. 定义：假设词表有 10 个词，每个词用 4 维向量表示
# 内部其实就是一个 10x4 的矩阵。刚创建时是随机的。
embed = nn.Embedding(num_embeddings=10, embedding_dim=4)

# 2. 输入：必须是单词的索引（LongTensor）
# 模拟：我想查第 0 号词和第 3 号词
word_indices = torch.LongTensor([0, 3])

# 3. 输出：查出来的向量
vectors = embed(word_indices)

print(f"权重矩阵形状: {embed.weight.shape}") # [10, 4]
print(f"输出向量内容:\n{vectors}")

权重矩阵形状: torch.Size([10, 4])
输出向量内容:
tensor([[ 0.7111, -1.4568,  0.2538, -0.1250],
        [ 0.3857, -1.0611, -1.2244, -0.3110]], grad_fn=<EmbeddingBackward0>)


Padding 填充

一个 Batch里的数据,形状必须是整齐的矩形。

但人类的句子长短不一，需要在短句子后填充特殊的 PAD 词。通常使用索引 `0` (记作 [PAD])

配套技术: 
  - `nn.Embedding` 的 `padding_idx`=0 参数。  这样索引为 0 的位置，其向量永远是全 0，且不参与梯度更新。
  
  - **Masking (掩码)**：告诉模型哪些是废话（0），哪些是真话（1）。


In [4]:
# 手写

import torch
import torch.nn as nn

# 1. 原始数据：三句长度不一的话
sentences = [
    "I love Pytorch",
    "Deep learning is amazing",
    "NLP is fun"
]
labels = torch.tensor([1, 1, 1]) # 假设都是正面情感

# 2. 【分词 Tokenization】：简单粗暴按空格切分
tokenized_sentences = [s.split() for s in sentences]
# 结果：[['I', 'love', 'Pytorch'], ['Deep', 'learning', 'is', 'amazing'], ['NLP', 'is', 'fun']]

# 3. 【构建词表 Vocab】：给每个词一个编号
# 我们需要两个特殊符号：<PAD>用于补齐，<UNK>用于处理没见过的词
vocab = {"<PAD>": 0, "<UNK>": 1, "I": 2, "love": 3, "Pytorch": 4, 
         "Deep": 5, "learning": 6, "is": 7, "amazing": 8, "NLP": 9, "fun": 10}

# 4. 【编号 Numericalization】：把单词转成数字索引
indexed_sentences = [[vocab.get(word, 1) for word in s] for s in tokenized_sentences]
# 结果：[[2, 3, 4], [5, 6, 7, 8], [9, 7, 10]]
#get方法  dict.get(key, default_value)


# 5. 【填充 Padding】：对齐长度
max_len = 4 # 统一对齐到最长的那句（4个词）
padded_sentences = []
for s in indexed_sentences:
    if len(s) < max_len:
        s = s + [0] * (max_len - len(s)) # 在后面补0
    else:
        s = s[:max_len] # 如果太长就截断
    padded_sentences.append(s)

# 转为张量 (Batch_Size=3, Seq_Len=4)
input_tensor = torch.LongTensor(padded_sentences) 

# ==========================================
# 6. 定义模型 (Embedding -> Linear)
# ==========================================
class SimpleNLPModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # Embedding层：padding_idx=0 告诉模型索引0只是填充，不计入学习
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # 全连接层：处理查表后的结果
        # 注意：Embedding输出是 [Batch, Seq_Len(每句话包含的词数), Embed_Dim]
        # nn.Linear 期望的输入通常是 [Batch, 特征数]
        # 我们先用 mean(维度1) 把一句话的所有词向量平均掉，变成 [Batch, Embed_Dim]
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [3, 4] (3句话，每句4个索引)
        embedded = self.embedding(x) # -> [3, 4, 8] (假设维度是8)
        
        # 将一句话里所有单词的特征融合（取平均）
        # 这一步叫 Global Average Pooling，把“序列”变成“一个整体特征”
        pooled = torch.mean(embedded, dim=1) # -> [3, 8]
        
        z = torch.relu(self.fc(pooled))
        return self.sigmoid(self.output(z))

# 7. 运行模型
model = SimpleNLPModel(vocab_size=len(vocab), embed_dim=8, hidden_dim=16)
predictions = model(input_tensor)

print(f"输入张量 (索引):\n{input_tensor}")
print(f"预测结果 (概率):\n{predictions}")

输入张量 (索引):
tensor([[ 2,  3,  4,  0],
        [ 5,  6,  7,  8],
        [ 9,  7, 10,  0]])
预测结果 (概率):
tensor([[0.4634],
        [0.4767],
        [0.4723]], grad_fn=<SigmoidBackward0>)


In [5]:
# 调用hugging_face transformers库 预处理

from transformers import AutoTokenizer

# 1. 加载一个现成的分词器 (以大模型 BERT 为例)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. 原始文本
sentences = ["I love Pytorch", "Deep learning is amazing"]

# 3. 一键完成：分词、编号、Padding、截断、转张量
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[  101,  1045,  2293,  1052, 22123,  2953,  2818,   102],
        [  101,  2784,  4083,  2003,  6429,   102,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0]])}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 1. 加载模型（以一个已经练好的情感分析模型为例）
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name) #加载预训练好的模型

# 2. 预处理
texts = ["I love this movie!", "This film is terrible."]
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# 3. 【核心步骤】喂给模型
with torch.no_grad(): # 推理模式，不计梯度
    outputs = model(**inputs) # ** 解包语法。是把字典里的 input_ids 等自动对号入座
    #等价于model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
print(outputs)


# 索引 0 对应 负面；索引 1 对应 正面
# 正数：模型觉得“是这一类”的可能性高于平均水平。
# 负数：模型觉得“是这一类”的可能性低于平均水平。

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

c:\Users\under\anaconda3\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\under\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

SequenceClassifierOutput(loss=None, logits=tensor([[-4.3246,  4.6837],
        [ 4.5167, -3.6952]]), hidden_states=None, attentions=None)
